# 04 Model Interpretability and HR Insights

Professional People Analytics notebook for Employee Attrition Prediction & Workforce Intelligence.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data" / "raw" / "WA_Fn-UseC_-HR-Employee-Attrition.csv").exists():
            return candidate
    raise RuntimeError("Project root not found. Open the notebook from the repository or one of its subfolders.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/Project/Data Science/employee_attrition_prediction_&_workforce_intelligence')

## Objective

Review final model interpretability artifacts, risk scores, and HR-friendly explanations. This notebook assumes the pipeline has been run and reads the generated outputs.

In [2]:
import pandas as pd

importance = pd.read_csv(PROJECT_ROOT / "outputs" / "metrics" / "permutation_importance.csv")
risk_scores = pd.read_csv(PROJECT_ROOT / "outputs" / "risk_scores" / "employee_attrition_risk_scores.csv")
lift = pd.read_csv(PROJECT_ROOT / "outputs" / "metrics" / "lift_gain_analysis.csv")

## Global Model Drivers

In [3]:
importance.head(15)

,feature,importance_mean,importance_std
0,overtime_flag,0.165857,0.021498
1,career_growth_score,0.099476,0.025587
2,NumCompaniesWorked,0.082887,0.012201
3,OverTime,0.080097,0.014275
4,JobLevel,0.058399,0.040300
5,engagement_score,0.056747,0.009366
6,JobSatisfaction,0.048867,0.012476
7,total_working_years_band,0.042773,0.025955
8,frequent_travel_flag,0.040822,0.020159
9,satisfaction_score,0.036476,0.008364


## Highest-Risk Review Queue

In [4]:
risk_scores.head(20)

,EmployeeNumber,Department,JobRole,OverTime,BusinessTravel,MonthlyIncome,YearsAtCompany,JobSatisfaction,EnvironmentSatisfaction,WorkLifeBalance,Attrition,attrition_flag,attrition_probability,risk_decile,risk_band,intervention_priority,risk_explanation
0,622,Research & Development,Laboratory Technician,Yes,Travel_Rarely,2340,1,4,3,1,Yes,1,0.998033,1,Critical,Priority 1 - executive review,Works overtime; Low work-life balance; Early t...
1,1494,Research & Development,Laboratory Technician,Yes,Travel_Frequently,3172,0,1,2,2,Yes,1,0.996902,1,Critical,Priority 1 - executive review,Works overtime; Frequent business travel; Low ...
2,167,Sales,Sales Representative,Yes,Travel_Rarely,1675,0,3,4,2,Yes,1,0.995980,1,Critical,Priority 1 - executive review,Works overtime; Low work-life balance; Early t...
3,1273,Sales,Sales Representative,Yes,Travel_Frequently,1118,1,4,3,3,Yes,1,0.995374,1,Critical,Priority 1 - executive review,Works overtime; Frequent business travel; Earl...
4,959,Sales,Sales Representative,Yes,Travel_Rarely,2121,1,2,4,4,Yes,1,0.995020,1,Critical,Priority 1 - executive review,Works overtime; Low job satisfaction; Early te...
5,614,Sales,Sales Representative,Yes,Travel_Frequently,1878,0,2,2,3,Yes,1,0.994865,1,Critical,Priority 1 - executive review,Works overtime; Frequent business travel; Low ...
6,952,Sales,Sales Representative,Yes,Travel_Rarely,2413,1,2,3,3,Yes,1,0.994054,1,Critical,Priority 1 - executive review,Works overtime; Low job satisfaction; Early te...
7,911,Research & Development,Laboratory Technician,Yes,Travel_Rarely,2795,1,4,1,1,Yes,1,0.993583,1,Critical,Priority 1 - executive review,Works overtime; Low environment satisfaction; ...
8,1504,Research & Development,Laboratory Technician,No,Travel_Frequently,2561,0,1,3,2,Yes,1,0.992686,1,Critical,Priority 1 - executive review,Frequent business travel; Low job satisfaction...
9,478,Sales,Sales Representative,Yes,Travel_Frequently,2174,3,2,1,3,Yes,1,0.992609,1,Critical,Priority 1 - executive review,Works overtime; Frequent business travel; Low ...


## Lift and Decile Evidence

In [5]:
lift

,risk_decile,employees,attrition_count,min_probability,max_probability,attrition_rate,lift,cumulative_attrition,cumulative_employees,cumulative_attrition_capture,cumulative_population_share,gain
0,1,29,17,0.823033,0.995374,0.586207,3.666911,17,29,0.361702,0.098639,0.263063
1,2,29,9,0.640303,0.822098,0.310345,1.941306,26,58,0.553191,0.197279,0.355913
2,3,30,8,0.459872,0.638518,0.266667,1.668085,34,88,0.723404,0.299320,0.424085
3,4,29,6,0.336351,0.447873,0.206897,1.294204,40,117,0.851064,0.397959,0.453105
4,5,30,2,0.253327,0.334224,0.066667,0.417021,42,147,0.893617,0.500000,0.393617
5,6,29,2,0.165021,0.252671,0.068966,0.431401,44,176,0.936170,0.598639,0.337531
6,7,29,0,0.103580,0.163596,0.000000,0.000000,44,205,0.936170,0.697279,0.238891
7,8,30,1,0.056539,0.099705,0.033333,0.208511,45,235,0.957447,0.799320,0.158127
8,9,29,1,0.025403,0.055623,0.034483,0.215701,46,264,0.978723,0.897959,0.080764
9,10,30,1,0.000925,0.023731,0.033333,0.208511,47,294,1.000000,1.000000,0.000000


## Group Monitoring Snapshots

In [6]:
fairness_frames = {
    "gender": pd.read_csv(PROJECT_ROOT / "outputs" / "metrics" / "fairness_by_gender.csv"),
    "department": pd.read_csv(PROJECT_ROOT / "outputs" / "metrics" / "fairness_by_department.csv"),
    "jobrole": pd.read_csv(PROJECT_ROOT / "outputs" / "metrics" / "fairness_by_jobrole.csv"),
}
fairness_frames["department"]

,Department,employees,actual_attrition_rate,mean_attrition_probability,high_risk_share
0,Research & Development,961,0.176899,0.347024,0.300728
1,Sales,446,0.136771,0.329673,0.271300
2,Human Resources,63,0.095238,0.254765,0.142857
